<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Threading;

// --- ИНТЕРФЕЙСЫ ---

// Базовый интерфейс для всех уведомлений
public interface INotificationSender
{
    void Send();
    string GetStatus();
}

// Интерфейс для уведомлений с приоритетом
public interface IPrioritizable
{
    int Priority { get; set; }
    bool IsHighPriority();
}

// --- БАЗОВЫЙ КЛАСС ---

// Базовый класс Notification с новыми атрибутами и методами
public class Notification : INotificationSender
{
    // Существующие поля
    private int _notificationId;
    private string _messageText;
    private string _type;
    private DateTime _createdDate;
    private bool _isRead;

    // --- Новые атрибуты ---
    private string _sender; // Отправитель
    private string _recipient; // Получатель
    private List<string> _tags = new List<string>(); // Теги для категоризации

    // Свойства для доступа к полям
    public int NotificationId { get => _notificationId; set => _notificationId = value; }
    public string MessageText
    {
        get => _messageText;
        set
        {
            if (string.IsNullOrEmpty(value))
                throw new ArgumentException("Сообщение не может быть пустым!");
            _messageText = value;
        }
    }
    public string Type { get => _type; protected set => _type = value; }
    public DateTime CreatedDate { get => _createdDate; private set => _createdDate = value; }
    public bool IsRead { get => _isRead; set => _isRead = value; }
    
    // --- Свойства для новых атрибутов ---
    public string Sender { get => _sender; set => _sender = value; }
    public string Recipient { get => _recipient; set => _recipient = value; }
    public List<string> Tags { get => _tags; }

    // Конструктор
    public Notification(int notificationId, string sender, string recipient, string messageText, string type)
    {
        NotificationId = notificationId;
        Sender = sender;
        Recipient = recipient;
        MessageText = messageText;
        Type = type;
        CreatedDate = DateTime.Now;
        IsRead = false;
    }

    // --- Новые методы ---
    public void AddTag(string tag)
    {
        if (!string.IsNullOrWhiteSpace(tag))
        {
            _tags.Add(tag.ToLower());
            Console.WriteLine($"К уведомлению {NotificationId} добавлен тег: '{tag.ToLower()}'.");
        }
    }

    public void MarkAsRead()
    {
        IsRead = true;
        Console.WriteLine($"Уведомление {NotificationId} отмечено как прочитанное.");
    }
    
    // --- РЕАЛИЗАЦИЯ ПОЛИМОРФИЗМА ---

    // 1. Перегрузка (Overloading) метода Send
    public virtual void Send()
    {
        Console.WriteLine($"[Отправка] Уведомление #{NotificationId} для {Recipient} от {Sender}.");
    }

    public virtual void Send(DateTime scheduledTime)
    {
        Console.WriteLine($"[Отложенная отправка] Уведомление #{NotificationId} будет отправлено {scheduledTime} для {Recipient}.");
    }

    // 2. Переопределение (Overriding)
    public virtual string GetStatus()
    {
        return $"Статус: {(IsRead ? "Прочитано" : "Не прочитано")}, Создано: {CreatedDate}";
    }

    public virtual void DisplayInfo()
    {
        Console.WriteLine($"ID: {NotificationId}, Тип: {Type}, От: {Sender}, Кому: {Recipient}");
        Console.WriteLine($"Сообщение: \"{MessageText}\"");
    }
}

// --- ПРОИЗВОДНЫЕ КЛАССЫ ---

public class EmailNotification : Notification, IPrioritizable
{
    // --- Новые атрибуты ---
    public string Subject { get; set; }
    public List<string> Attachments { get; private set; } = new List<string>();
    public int Priority { get; set; }

    public EmailNotification(int id, string sender, string recipientEmail, string subject, string message)
        : base(id, sender, recipientEmail, message, "Email")
    {
        Subject = subject;
        Priority = 5; // Приоритет по умолчанию
    }

    public void AddAttachment(string fileName)
    {
        Attachments.Add(fileName);
        Console.WriteLine($"К письму '{Subject}' добавлен файл: {fileName}");
    }

    public bool IsHighPriority() => Priority > 8;

    public override void Send()
    {
        Console.WriteLine($"[Отправка Email] Кому: {Recipient}, Тема: {Subject}");
        if (Attachments.Count > 0)
            Console.WriteLine($"С {Attachments.Count} вложениями.");
        if (IsHighPriority())
            Console.WriteLine("ВАЖНОЕ ПИСЬМО!");
    }
}

public class SmsNotification : Notification
{
    // --- Новые атрибуты ---
    public string Operator { get; private set; }
    private bool _deliveryReportRequired;

    public SmsNotification(int id, string sender, string recipientPhone, string message, string mobileOperator)
        : base(id, sender, recipientPhone, message, "SMS")
    {
        Operator = mobileOperator;
        _deliveryReportRequired = false;
    }

    // --- Новый метод ---
    public void RequestDeliveryReport()
    {
        _deliveryReportRequired = true;
        Console.WriteLine("Запрошен отчет о доставке SMS.");
    }
    
    // --- Переопределение метода ---
    public override void Send()
    {
        Console.WriteLine($"[Отправка SMS] На номер {Recipient} через оператора {Operator}.");
        if(_deliveryReportRequired)
             Console.WriteLine("Будет запрошен статус доставки.");
    }
}

public class PushNotification : Notification, IPrioritizable
{
    // --- Новые атрибуты ---
    public string Platform { get; set; } // 'iOS', 'Android', 'Web'
    public string Sound { get; set; }
    public int Priority { get; set; }

    public PushNotification(int id, string sender, string recipientDeviceId, string message, string platform)
        : base(id, sender, recipientDeviceId, message, "Push")
    {
        Platform = platform;
        Sound = "default";
        Priority = 5;
    }

    public bool IsHighPriority() => Priority > 7;

    // --- Переопределение метода ---
    public override void Send()
    {
        Console.WriteLine($"[Отправка Push] На платформу {Platform} (устройство: {Recipient}).");
        Console.WriteLine($"Звук: {Sound}. Приоритет: {Priority}.");
    }

    // --- Переопределение перегруженного метода ---
    public override void Send(DateTime scheduledTime)
    {
        Console.WriteLine($"[Отложенный Push] Уведомление для {Platform} будет отправлено в {scheduledTime}.");
    }
}

// --- GENERIC КЛАСС ---

// Универсальный класс для управления очередью уведомлений
public class NotificationQueue<T> where T : INotificationSender
{
    private Queue<T> _queue = new Queue<T>();

    public int Count => _queue.Count;

    // Добавление уведомления в очередь
    public void Enqueue(T notification)
    {
        Console.WriteLine($"Уведомление типа {notification.GetType().Name} добавлено в очередь.");
        _queue.Enqueue(notification);
    }

    // Отправка следующего уведомления в очереди
    public void ProcessNext()
    {
        if (_queue.Count > 0)
        {
            T notification = _queue.Dequeue();
            Console.WriteLine($"--- Обработка уведомления из очереди ---");
            notification.Send();
            Console.WriteLine("-------------------------------------");
        }
        else
        {
            Console.WriteLine("Очередь уведомлений пуста.");
        }
    }
}


// --- ДЕМОНСТРАЦИЯ ---

// 1. Создание объектов с новыми атрибутами
var email = new EmailNotification(101, "system@corp.com", "user@example.com", "Квартальный отчет", "Отчет во вложении.");
email.AddTag("work");
email.AddTag("reports");
email.AddAttachment("report_q3.pdf");
email.Priority = 9; // Высокий приоритет

var sms = new SmsNotification(102, "BANK", "+79991234567", "Код подтверждения: 5566", "MTS");
sms.AddTag("security");
sms.RequestDeliveryReport();

var push = new PushNotification(103, "app.games", "device-token-xyz", "Ваша энергия восстановлена!", "Android");
push.Sound = "notification.mp3";
push.Priority = 8;

Console.WriteLine("\n=== Демонстрация полиморфизма (перегрузка и переопределение) ===\n");

Notification[] notifications = { email, sms, push };

foreach (var n in notifications)
    {
        n.DisplayInfo();
            
        // Вызов обычного метода Send() (переопределенного в каждом классе)
        n.Send(); 
            
        // Вызов перегруженного метода Send()
        n.Send(DateTime.Now.AddMinutes(30)); 
            
        Console.WriteLine(n.GetStatus());
        Console.WriteLine("---");
    }

Console.WriteLine("\n=== Демонстрация Generic класса NotificationQueue ===\n");
        
// Создаем очередь для всех типов уведомлений
var notificationProcessor = new NotificationQueue<Notification>();
        
// Добавляем уведомления в очередь
notificationProcessor.Enqueue(email);
notificationProcessor.Enqueue(sms);
notificationProcessor.Enqueue(push);
        
Console.WriteLine($"\nВ очереди {notificationProcessor.Count} уведомления. Начинаем обработку...");
        
// Обрабатываем все уведомления по очереди
notificationProcessor.ProcessNext();
notificationProcessor.ProcessNext();
notificationProcessor.ProcessNext();
notificationProcessor.ProcessNext(); // Попытка обработать пустую очередь


К уведомлению 101 добавлен тег: 'work'.
К уведомлению 101 добавлен тег: 'reports'.
К письму 'Квартальный отчет' добавлен файл: report_q3.pdf
К уведомлению 102 добавлен тег: 'security'.
Запрошен отчет о доставке SMS.

=== Демонстрация полиморфизма (перегрузка и переопределение) ===

ID: 101, Тип: Email, От: system@corp.com, Кому: user@example.com
Сообщение: "Отчет во вложении."
[Отправка Email] Кому: user@example.com, Тема: Квартальный отчет
С 1 вложениями.
ВАЖНОЕ ПИСЬМО!
[Отложенная отправка] Уведомление #101 будет отправлено 10/19/2025 6:11:07 PM для user@example.com.
Статус: Не прочитано, Создано: 10/19/2025 5:41:07 PM
---
ID: 102, Тип: SMS, От: BANK, Кому: +79991234567
Сообщение: "Код подтверждения: 5566"
[Отправка SMS] На номер +79991234567 через оператора MTS.
Будет запрошен статус доставки.
[Отложенная отправка] Уведомление #102 будет отправлено 10/19/2025 6:11:07 PM для +79991234567.
Статус: Не прочитано, Создано: 10/19/2025 5:41:07 PM
---
ID: 103, Тип: Push, От: app.games, Кому